In [1]:
import matplotlib
import platform
print ("Operating system: ", platform.system())
if "Linux" in platform.system():
    %matplotlib tk
else:
    %matplotlib qt

    
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

#
import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np

#
import scipy
import os
import time

#
import pickle 


#
from calibration.CalibrationTools import (CalibrationTools, get_binary_std_map, get_rois_stardist2d, get_img_std,
                                          save_calibration_data, compute_roi_traces_f0_alignment, align_to_prev_day)

from drift.drift import (make_template, compute_drift_multi_frames, correct_drift, 
                         correct_drift_single_frame, template_generation, 
                         plot_mean_vs_template, make_motion_template_and_correct_data)
from utils.utils import smooth_ca_time_series, compute_dff0



Operating system:  Linux


Autosaving every 180 seconds


2023-01-18 16:59:09.425247: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-01-18 16:59:09.505688: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-01-18 16:59:09.507305: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/cat/.conda/envs/bmi/lib/python3.8/site-packages/cv2/../../lib64:
2023-01-18 16:59:09.507

In [2]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################

#
fname_calibration_today = '/media/cat/4TB/donato/DON-010795/22-07-30/calibration/Image_001_001.raw'
fname_rois_previous_day = '/media/cat/4TB/donato/DON-010795/22-07-30/rois_pixels_and_thresholds.npz'

# 
bmi_c = CalibrationTools(fname_calibration_today)
bmi_c.align_data = np.load(fname_rois_previous_day,
                            allow_pickle=True)
#
bmi_c.smooth_ca_time_series = smooth_ca_time_series
bmi_c.compute_dff0 = compute_dff0

#
bmi_c.subsample = 10 # for std computation downsample to every N'th frame; the more frames the better the rois;
                  #   TODO: use correlation instead?! might be much faster; it is fast in other implemenations
    
print ("DONE...")

memmap :  (20000, 512, 512)
DONE...


In [3]:
###############################################
############### GET TEMPLATE ##################
###############################################
#
bmi_c.template = np.mean(bmi_c.data[:5000],axis=0)
bmi_c.template = bmi_c.template - np.mean(bmi_c.template)
bmi_c.std_map = bmi_c.template
#bmi_c.std_map = bmi_c.filter_data_make_std_map()
print ("...DONE...")



...DONE...


In [4]:
#############################################################
############# ALIGN TO PREVIOUS RECORDING ###################
#############################################################

#
bmi_c = align_to_prev_day(bmi_c)

#
bmi_c.vmin = 0
bmi_c.vmax = 100

# visualize traces
bmi_c.compute_traces_ensembles(bmi_c.template)

#
print ("DONE...")


KeyError: 'contours_all_cells is not a file in the archive'

In [8]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################
# 
bmi_c.sample_rate = 30
bmi_c.post_reward_lockout = 10   # reward lockout in seconds
                                 # TODO: in future load/save this to disk so that BMI 
                                 #   - doesn't use differetn lockout than calibration step
bmi_c.balance_ensemble_rewards_flag = False   #this makes sure that both ensembles elicit a similar number of random rewards
bmi_c.rois_smooth_window = 5     # of frames to use to smooth the realtime signal
bmi_c.smooth_diff_function_flag = True    # use a kernel window to smooth current value

#
# find 30% reward threshold
bmi_c.reward_rate = 0.5#*0.85

#gr.find_reward_thresholds_low_and_high()
bmi_c.find_reward_thresholds_high()  # this only rewards when sound passes specific level

#
print ("thresholds: ", bmi_c.high)

############################################## 
bmi_c.plot_rewarded_ensembles()


#
bmi_c.reward_rate_scaling_factor = 1.0

#
bmi_c.high = bmi_c.high*bmi_c.reward_rate_scaling_factor
print ("bmi_c.high: scaled by: ", bmi_c.reward_rate_scaling_factor, ", final value:  ", bmi_c.high)


COMPUTED # of roi traces:  19


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 19995/19995 [00:00<00:00, 1754867.30it/s]

low, high:  -1.3170082119297615 1.2861344307348712
nsec recording:  666 max # of random rewards (i.e. every 30sec)  22
 @30% reward:  11
updated rewards #:  11 -0.5796504369036642 0.5660620624376735
thresholds:  0.5660620624376735
bmi_c.high: scaled by:  1.0 , final value:   0.5660620624376735


In [19]:
#############################################
########### SAVE DATA #######################
#############################################

#    
save_calibration_data(bmi_c)  

[autoreload of calibration.CalibrationTools failed: Traceback (most recent call last):
  File "/home/cat/.conda/envs/bmi/lib/python3.8/site-packages/IPython/extensions/autoreload.py", line 261, in check
    superreload(m, reload, self.old_objects)
  File "/home/cat/.conda/envs/bmi/lib/python3.8/site-packages/IPython/extensions/autoreload.py", line 459, in superreload
    module = reload(module)
  File "/home/cat/.conda/envs/bmi/lib/python3.8/importlib/__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 604, in _exec
  File "<frozen importlib._bootstrap_external>", line 843, in exec_module
  File "<frozen importlib._bootstrap>", line 219, in _call_with_frames_removed
  File "/home/cat/code/bmi/calibration/CalibrationTools.py", line 5, in <module>
    multiprocessing.set_start_method('spawn')
  File "/home/cat/.conda/envs/bmi/lib/python3.8/multiprocessing/context.py", line 243, in set_start_method
    raise RuntimeError('conte

TypeError: cannot pickle '_io.BufferedReader' object

In [49]:
data = np.load('/media/cat/4TBSSD/donato/DON8498/22-06-08/rois_pixels_and_thresholds.npz',
               allow_pickle=True)

ensemble1_footprints = data['ensemble1_footprints']
contours_local = data['ensemble1_contours']

FileNotFoundError: [Errno 2] No such file or directory: '/media/cat/4TBSSD/donato/DON8498/22-06-08/rois_pixels_and_thresholds.npz'

In [4]:
cell_id =1
plt.figure()
temp = bmi_c.roi_traces[cell_id]
plt.plot((temp - np.median(temp))/np.median(temp))
plt.plot((temp - bmi_c.roi_f0s[cell_id])/bmi_c.roi_f0s[cell_id])
plt.show()

NameError: name 'bmi_c' is not defined

In [6]:
plt.figure()
for c in range(len(contours_local)):
    for k in range(len(contours_local[c])-1):

        plt.plot([contours_local[c][k][0],
                            contours_local[c][k+1][0]],
                           [contours_local[c][k][1],
                            contours_local[c][k+1][1]],
                            c='blue',
                            linewidth=5)
plt.show()


In [16]:
data1 = np.load('/media/cat/4TB1/donato/DON-010800/22-08-15/data/results_regular_filter.npz',
                allow_pickle=True)
e1 = data1['ensemble_activity']
print (e1.shape)


data2 = np.load('/media/cat/4TB1/donato/DON-010800/22-08-15/data/results_5timesteps.npz',
                allow_pickle=True)
e2 = data2['ensemble_activity']
print (e2.shape)

data3 = np.load('/media/cat/4TB1/donato/DON-010800/22-08-15/data/results_no_filter.npz',
                allow_pickle=True)
e3 = data3['ensemble_activity']
print (e3.shape)

data4 = np.load('/media/cat/4TB1/donato/DON-010800/22-08-15/data/results.npz',
                allow_pickle=True)
e4 = data4['ensemble_activity']
print (e4.shape)


#############################################################
plt.figure()

t = np.arange(e1.shape[1])/30
plt.plot(t,
         e1[0]-e1[1],
         c='blue', 
         linewidth = 4,
         label='current filter')
plt.plot(t,
         e2[0]-e2[1],
         linewidth = 4,
         c='red', label = 'mean last 5 time steps')
plt.plot(t,
         e3[0]-e3[1],
         linewidth = 4,
         c='green', label = 'no filter')
plt.plot(t,
         e4[0]-e4[1],
         linewidth = 4,
         c='black', label = 'graded filter last 5 time steps')


plt.xlabel("time (sec)", fontsize=16)
plt.ylabel("Ensemble 1 - Ensemble 2", fontsize=16)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.plot([t[0],t[-1]],
         [0,0],
         '--',
        c='black')
plt.legend()
plt.show()

(2, 3000)
(2, 3000)
(2, 3000)
(2, 3000)


In [3]:
def get_octave_frequencies(low_freq,
						   high_freq,
						   octave_size=0.25):
	#
	octaves = []

	#
	octaves.append(low_freq)
	temp = low_freq
	while True:
		temp = temp * (1 + octave_size)
		if temp > high_freq:
			break
		octaves.append(temp)
	"""
	low_freq = 1000
	high_freq = 16000
	octaves = np.arange(int(low_freq/1000), 1+int(high_freq/1000))
    
	octaves = 2**(octave_size*x)
    
	#
	return np.array(1000*octaves)
	"""
	octaves = np.arange(int(low_freq/1000), 1+int(high_freq/1000))
	octaves = 2**(octave_size*x)
	return np.array(1000*octaves)

	#

In [4]:
low_freq = 2000
high_freq = 16000

octaves = get_octave_frequencies(low_freq,high_freq,octave_size=0.25)

NameError: name 'x' is not defined